# Newton & quasi-Newton methods (BFGS, L-BFGS) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Use curvature to rescale the gradient before stepping
Newton methods ask not only which way is downhill but how curved each direction is. These five examples show Newton solving a quadratic in one step, a damped Newton path, BFGS satisfying the secant equation, positive curvature preserving positive definiteness, and L-BFGS-style memory approximating a direction.

In [ ]:
# 1) Newton solves a quadratic in one step
H=np.array([[4.,1.],[1.,3.]]); b=np.array([1.,2.]); x=np.array([0.,0.]); grad=H@x-b; step=-np.linalg.solve(H,grad); xn=x+step; xstar=np.linalg.solve(H,b)
plt.figure(figsize=(5,4)); plt.scatter([x[0],xn[0]],[x[1],xn[1]],c=['r','g']); plt.plot([x[0],xn[0]],[x[1],xn[1]],'--'); plt.title('one Newton step reaches optimum')
assert np.allclose(xn,xstar)

In [ ]:
# 2) damped Newton tames a nonlinear step
f=lambda x:x**4+0.5*x**2; gp=lambda x:4*x**3+x; hp=lambda x:12*x**2+1; x=2.; path=[x]
for _ in range(6): x=x-0.5*gp(x)/hp(x); path.append(x)
plt.figure(figsize=(5,3)); plt.plot(path,'-o'); plt.title('damped Newton')
assert abs(path[-1])<abs(path[0])

In [ ]:
# 3) BFGS update satisfies B s = y
B=np.eye(2); s=np.array([1.,2.]); y=np.array([3.,5.]); Bs=B@s; Bnew=B - np.outer(Bs,Bs)/(s@Bs) + np.outer(y,y)/(y@s)
plt.figure(figsize=(5,3)); plt.bar(['secant residual'],[np.linalg.norm(Bnew@s-y)]); plt.title('BFGS matches new curvature pair')
assert np.linalg.norm(Bnew@s-y)<1e-12

In [ ]:
# 4) positive curvature keeps B positive definite
B=np.eye(2); s=np.array([1.,1.]); y=np.array([2.,3.]); Bs=B@s; Bnew=B - np.outer(Bs,Bs)/(s@Bs) + np.outer(y,y)/(y@s); eig=np.linalg.eigvalsh(Bnew)
plt.figure(figsize=(5,3)); plt.bar(['lambda_min','lambda_max'],eig); plt.title('positive eigenvalues after BFGS')
assert s@y>0 and eig.min()>0

In [ ]:
# 5) two-loop L-BFGS direction with two stored pairs
pairs=[(np.array([1.,0.]),np.array([2.,0.])),(np.array([0.,1.]),np.array([0.,4.]))]; q=np.array([2.,4.]); al=[]
for s,y in pairs[::-1]:
    rho=1/(y@s); a=rho*(s@q); al.append(a); q=q-a*y
r=q
for (s,y),a in zip(pairs,al[::-1]):
    rho=1/(y@s); beta=rho*(y@r); r=r+s*(a-beta)
plt.figure(figsize=(5,4)); plt.arrow(0,0,-r[0],-r[1],head_width=.08); plt.title('L-BFGS inverse-Hessian direction')
assert np.allclose(r,[1.,1.])